# Paper-to-Project: Phase 1–6 Multi-Paper Integration Test

This notebook runs the **complete backend pipeline** over **all 29 research papers** and saves a consolidated report.

| Phase | Day(s) | Description |
|-------|--------|-------------|
| Phase 1 | Day 1–4 | PDF Extraction & Multi-Engine Routing |
| Phase 2 | Day 5–8 | Canonical PaperDocument Merge & Section Audit |
| Phase 3 | Day 9–13 | Confidence, Provenance & Ingestion Benchmarks |
| Phase 4 | Day 14–18 | Semantic Chunking, Embeddings & RAG Retrieval |
| Phase 5 | Day 19–22 | Ingestion, Decomposition, Parameters, Gap Finding |
| Phase 6 | Day 23–27 | Hardware profiler, Resource estimation, Feasibiltiy Engine, Refinement, sequencing |

## Cell 0: Environment Setup — Discover All PDFs

In [1]:
import os
import sys
import json
import time
import datetime
import traceback
from collections import Counter

# ---- PATH SETUP ----
NOTEBOOK_DIR = os.path.abspath('')
if os.path.basename(NOTEBOOK_DIR) == 'tests':
    BACKEND_DIR = os.path.dirname(NOTEBOOK_DIR)
else:
    BACKEND_DIR = NOTEBOOK_DIR

if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)

PAPERS_DIR  = os.path.join(BACKEND_DIR, 'papers', 'research_papers')
REPORTS_DIR = os.path.join(BACKEND_DIR, 'tests', 'reports')
os.makedirs(REPORTS_DIR, exist_ok=True)

# ---- DISCOVER ALL PDFs ----
all_pdfs = sorted(
    [os.path.join(PAPERS_DIR, f) for f in os.listdir(PAPERS_DIR) if f.lower().endswith('.pdf')],
    key=lambda p: int(os.path.basename(p).replace('[', '').replace('].pdf', ''))
)

print(f'[ENV] Backend dir : {BACKEND_DIR}')
print(f'[ENV] Papers dir  : {PAPERS_DIR}')
print(f'[ENV] Reports dir : {REPORTS_DIR}')
print(f'[ENV] PDFs found  : {len(all_pdfs)}')
print()
for i, p in enumerate(all_pdfs, 1):
    print(f'  [{i:>2}] {os.path.basename(p)}')

[ENV] Backend dir : c:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\backend
[ENV] Papers dir  : c:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\backend\papers\research_papers
[ENV] Reports dir : c:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\backend\tests\reports
[ENV] PDFs found  : 29

  [ 1] [1].pdf
  [ 2] [2].pdf
  [ 3] [3].pdf
  [ 4] [4].pdf
  [ 5] [5].pdf
  [ 6] [6].pdf
  [ 7] [7].pdf
  [ 8] [8].pdf
  [ 9] [9].pdf
  [10] [10].pdf
  [11] [11].pdf
  [12] [12].pdf
  [13] [13].pdf
  [14] [14].pdf
  [15] [15].pdf
  [16] [16].pdf
  [17] [17].pdf
  [18] [18].pdf
  [19] [19].pdf
  [20] [20].pdf
  [21] [21].pdf
  [22] [22].pdf
  [23] [23].pdf
  [24] [24].pdf
  [25] [25].pdf
  [26] [26].pdf
  [27] [27].pdf
  [28] [28].pdf
  [29] [29].pdf


## Cell 1: Auto-Detect System Configuration

All values are **auto-detected** from your machine:
- **Model** → queried from the local Ollama REST API
- **GPU / VRAM** → detected via `torch.cuda` then `nvidia-smi` fallback
- **System RAM** → detected via `psutil`
- **Timeline** → only value you need to set manually

In [2]:
import subprocess
import requests

# ================================================================
# AUTO-DETECT: Ollama Model
# Queries the local Ollama REST API and picks the best available.
# ================================================================
PREFERRED_MODELS = [
    'qwen2.5-coder:1.5b', 'qwen2.5-coder:7b',
    'llama3', 'mistral', 'gemma'
]
MODEL_NAME = None
try:
    resp = requests.get('http://localhost:11434/api/tags', timeout=5)
    available_models = [m['name'] for m in resp.json().get('models', [])]
    for pref in PREFERRED_MODELS:
        if pref in available_models:
            MODEL_NAME = pref
            break
    if not MODEL_NAME and available_models:
        MODEL_NAME = available_models[0]
    print(f'[AUTO] Ollama models available : {available_models}')
    print(f'[AUTO] Selected model          : {MODEL_NAME}')
except Exception as e:
    MODEL_NAME = 'qwen2.5-coder:1.5b'
    print(f'[WARN] Ollama API unreachable ({e}). Fallback: {MODEL_NAME}')

# ================================================================
# AUTO-DETECT: System RAM via psutil
# ================================================================
try:
    import psutil
    system_ram_gb = round(psutil.virtual_memory().total / (1024 ** 3), 1)
    print(f'[AUTO] System RAM              : {system_ram_gb} GB')
except ImportError:
    system_ram_gb = 16.0
    print(f'[WARN] psutil not installed. Defaulting RAM = {system_ram_gb} GB')

# ================================================================
# AUTO-DETECT: GPU Name + VRAM
# Tries torch.cuda first, then nvidia-smi as fallback.
# ================================================================
gpu_model = 'CPU (no GPU detected)'
vram_gb   = 0.0
try:
    import torch
    if torch.cuda.is_available():
        gpu_model = torch.cuda.get_device_name(0)
        vram_gb   = round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 1)
        print(f'[AUTO] GPU  (torch)            : {gpu_model}')
        print(f'[AUTO] VRAM (torch)            : {vram_gb} GB')
    else:
        raise RuntimeError('CUDA not available in torch')
except Exception as torch_err:
    print(f'[WARN] torch CUDA failed ({torch_err}). Trying nvidia-smi...')
    try:
        smi_name = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            timeout=5, text=True
        ).strip().splitlines()[0]
        smi_mem = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
            timeout=5, text=True
        ).strip().splitlines()[0]
        gpu_model = smi_name
        vram_gb   = round(int(smi_mem) / 1024, 1)   # MiB -> GiB
        print(f'[AUTO] GPU  (nvidia-smi)       : {gpu_model}')
        print(f'[AUTO] VRAM (nvidia-smi)       : {vram_gb} GB')
    except Exception as smi_err:
        print(f'[WARN] nvidia-smi failed ({smi_err}). GPU constraints = 0.')

# ================================================================
# MANUAL: Only timeline needs human input
# ================================================================
TIMELINE_WEEKS = 2   # <-- change this if needed

CONSTRAINTS = {
    'gpu_model'      : gpu_model,
    'system_ram_gb'  : system_ram_gb,
    'vram_gb'        : vram_gb,
    'timeline_weeks' : TIMELINE_WEEKS
}

# ================================================================
# SCOPE: Set PAPER_LIMIT to None for all 29, or an integer
# (e.g. 3) for a quick test before the full run.
# ================================================================
PAPER_LIMIT   = None   # None = all papers
SKIP_ON_ERROR = True   # True = log errors and continue

papers_to_run = all_pdfs[:PAPER_LIMIT] if PAPER_LIMIT else all_pdfs

print()
print('--- Final Configuration ---')
print(f'  Model          : {MODEL_NAME}')
print(f'  GPU            : {gpu_model}')
print(f'  VRAM           : {vram_gb} GB')
print(f'  System RAM     : {system_ram_gb} GB')
print(f'  Timeline       : {TIMELINE_WEEKS} weeks')
print(f'  Papers to run  : {len(papers_to_run)}')
print(f'  Skip on error  : {SKIP_ON_ERROR}')

[AUTO] Ollama models available : ['nomic-embed-text:latest', 'qwen2.5-coder:1.5b']
[AUTO] Selected model          : qwen2.5-coder:1.5b
[AUTO] System RAM              : 23.6 GB
[WARN] torch CUDA failed (CUDA not available in torch). Trying nvidia-smi...
[AUTO] GPU  (nvidia-smi)       : NVIDIA GeForce RTX 5050 Laptop GPU
[AUTO] VRAM (nvidia-smi)       : 8.0 GB

--- Final Configuration ---
  Model          : qwen2.5-coder:1.5b
  GPU            : NVIDIA GeForce RTX 5050 Laptop GPU
  VRAM           : 8.0 GB
  System RAM     : 23.6 GB
  Timeline       : 2 weeks
  Papers to run  : 29
  Skip on error  : True


## Cell 2: Import Pipeline Orchestrator

In [3]:
from pipeline import graph as orchestrator
print('[OK] Pipeline orchestrator imported successfully.')

[OK] Pipeline orchestrator imported successfully.


## Cell 3: Run Pipeline Over All Papers

> Processes every paper sequentially with the full Phase 1–5 treatment.  
> Results are collected into `all_results` and failures into `all_errors`.

In [4]:
all_results = []
all_errors = []

run_start_time = time.time()
RUN_TS = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

print("=" * 65)
print(f"RUNNING COMPLETE PHASE 1-7 PIPELINE FOR {len(papers_to_run)} PAPERS")
print(f"Model: {MODEL_NAME} | GPU: {CONSTRAINTS['gpu_model']}")
print("=" * 65)

for paper_idx, pdf_path in enumerate(papers_to_run, 1):
    pdf_name = os.path.basename(pdf_path)
    paper_id = f"paper_{pdf_name.replace('[', '').replace('].pdf', '')}"

    print()
    print("=" * 65)
    print(f"[{paper_idx:>2}/{len(papers_to_run)}] Processing: {pdf_name}")
    print("=" * 65)

    initial_state = {
        'pdf_path': pdf_path,
        'constraints': CONSTRAINTS,
        'model_name': MODEL_NAME,
        'loop_count': 0
    }

    t0 = time.time()
    try:
        # Run graph (now automatically executes ingestion, sequencing, spec, and code generation)
        result = orchestrator.invoke(initial_state)
        elapsed = round(time.time() - t0, 2)

        meta = result.get('metadata')
        pdoc = result.get('paper_doc')
        cg = result.get('component_graph')
        ext_params = result.get('extracted_parameters')
        gap_rpt = result.get('gap_report')
        res_est = result.get('resource_estimation')
        feat = result.get('feasibility_report')
        bseq = result.get('build_sequence')
        spec = result.get('project_specification')
        tree = result.get('project_tree')
        arpt = result.get('report')

        gap_counts = Counter(g.classification for g in gap_rpt.parameter_gaps) if gap_rpt else {}
        param_status_counts = dict(Counter(
            getattr(ext_params, f).status for f in ext_params.__class__.model_fields.keys()
        )) if ext_params else {}

        # Capture adaptations
        adaptations = []
        if cg:
            for comp in cg.components:
                for param_name, param_details in comp.parameters.items():
                    if "PAPER ORIGINAL" in param_details.rationale:
                        adaptations.append({
                            'component': comp.name,
                            'parameter': param_name,
                            'value': param_details.value,
                            'trace': param_details.rationale
                        })

        paper_result = {
            'paper_id': paper_id,
            'pdf_name': pdf_name,
            'status': 'SUCCESS',
            'elapsed_seconds': elapsed,
            'title': meta.title if meta else 'Unknown Remote Sensing Research Paper',
            'authors': meta.authors if meta else [],
            'abstract': meta.abstract if meta else '',
            'primary_contribution': meta.primary_contribution if meta else '',
            'sections_count': len(pdoc.sections) if pdoc else 0,
            'tables_count': len(pdoc.tables) if pdoc else 0,
            'equations_count': len(pdoc.equations) if pdoc else 0,
            'components_count': len(cg.components) if cg else 0,
            'edges_count': len(cg.edges) if cg else 0,
            'param_status_counts': param_status_counts,
            'gap_counts': dict(gap_counts),
            'feasibility_status': feat.overall_status if feat else 'UNKNOWN',
            'milestones_count': len(bseq.milestones) if bseq else 0,
            'total_duration_weeks': getattr(bseq, 'total_duration_weeks', 0.0) if bseq else 0.0,
            'adaptations': adaptations,
            'generated_files_count': len(tree.files) if tree else 0,
            '_result_full': result
        }

        all_results.append(paper_result)
        print(f"  [OK] Done in {elapsed}s")

    except Exception as e:
        elapsed = round(time.time() - t0, 2)
        err_msg = str(e)
        print(f"  [ERROR] {pdf_name} failed after {elapsed}s: {err_msg}")
        err_data = {
            'paper_id': paper_id,
            'pdf_name': pdf_name,
            'status': 'ERROR',
            'elapsed_seconds': elapsed,
            'error': err_msg,
            'traceback': traceback.format_exc()
        }
        all_errors.append(err_data)



total_run_time = round(time.time() - run_start_time, 2)
print()
print('=' * 65)
print(f'RUN COMPLETE: {len(all_results)} success, {len(all_errors)} errors')
print(f'Total time  : {total_run_time}s ({round(total_run_time/60, 1)} min)')
print('=' * 65)

RUNNING COMPLETE PHASE 1-7 PIPELINE FOR 29 PAPERS
Model: qwen2.5-coder:1.5b | GPU: NVIDIA GeForce RTX 5050 Laptop GPU

[ 1/29] Processing: [1].pdf


c:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



[Orchestrator] Step 1: Parsing and merging paper 'c:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\backend\papers\research_papers\[1].pdf'...
[2026-08-24 22:28:20] [INFO] [paper_to_project] 🚀 Starting routed extraction pipeline for '[1].pdf'...
[2026-08-24 22:28:20] [INFO] [paper_to_project] Routing '[1].pdf' to PyMuPDF text & section parser.
[2026-08-24 22:28:20] [INFO] [paper_to_project] Extracting blocks from PDF '[1].pdf' using two_column layout...
[2026-08-24 22:28:29] [INFO] [paper_to_project] Detecting sections for paper: 'A Novel Change Detection Method Based on Visual'...
[2026-08-24 22:28:29] [INFO] [paper_to_project] Section pruning triggered by header: 'REFERENCES'
[2026-08-24 22:28:29] [INFO] [paper_to_project] Checking GROBID server availability at http://localhost:8070...
[2026-08-24 22:28:33] [WARNING] [paper_to_project] GROBID is offline at http://localhost:8070. Engaging failover to Docling for '[1].pdf'.
[2026-08-24 22:2

[INFO] 2026-08-24 22:28:35,318 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-24 22:28:35,329 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-24 22:28:35,329 [RapidOCR] main.py:63: Using C:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-24 22:28:35,415 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-24 22:28:35,417 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-24 22:28:35,417 [RapidOCR] main.py:63: Using C:\Users\kvcsu_ht23nk8\OneDrive\Desktop\al

[2026-08-24 22:29:23] [INFO] [paper_to_project] Docling conversion completed successfully for '[1].pdf'.
[2026-08-24 22:29:23] [INFO] [paper_to_project] [FINISH] Finished routed extraction for '[1].pdf'. Selected: ['pymupdf', 'docling']
[2026-08-24 22:29:23] [INFO] [paper_to_project] Merging extraction outputs for '[1].pdf' into canonical PaperDocument...
[Orchestrator] Slicing document and saving embeddings in pgvector database...
[DB WARN] PostgreSQL database unreachable. Falling back to local JSON storage: c:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\backend\papers\in_memory_vector_db.json
[DB] Local JSON database initialized successfully (Fallback).
[DB] Saved 20 chunks with local vector lists for 'paper_1'.
Initializing ChatOllama with model 'qwen2.5-coder:1.5b'...
Sending request to local Ollama for structured metadata extraction...

[Orchestrator] Step 2: Running Method Decomposition Agent...
[Decomposition Agent] Querying local 

...

[2026-08-25 01:02:59] [INFO] [paper_to_project] Docling conversion completed successfully for '[27].pdf'.
[2026-08-25 01:02:59] [INFO] [paper_to_project] [FINISH] Finished routed extraction for '[27].pdf'. Selected: ['pymupdf', 'docling']
[2026-08-25 01:02:59] [INFO] [paper_to_project] Merging extraction outputs for '[27].pdf' into canonical PaperDocument...
[Orchestrator] Slicing document and saving embeddings in pgvector database...
[DB WARN] PostgreSQL database unreachable. Falling back to local JSON storage: c:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\backend\papers\in_memory_vector_db.json
[DB] Local JSON database initialized successfully (Fallback).
[DB] Saved 7 chunks with local vector lists for 'paper_27'.
Initializing ChatOllama with model 'qwen2.5-coder:1.5b'...
Sending request to local Ollama for structured metadata extraction...

[Orchestrator] Step 2: Running Method Decomposition Agent...
[Decomposition Agent] Querying loc

RapidOCR returned empty result!
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


[2026-08-25 01:10:07] [INFO] [paper_to_project] Docling conversion completed successfully for '[28].pdf'.
[2026-08-25 01:10:07] [INFO] [paper_to_project] [FINISH] Finished routed extraction for '[28].pdf'. Selected: ['pymupdf', 'docling']
[2026-08-25 01:10:07] [INFO] [paper_to_project] Merging extraction outputs for '[28].pdf' into canonical PaperDocument...
  [ERROR] [28].pdf failed after 52.72s: cannot access local variable 'Equation' where it is not associated with a value

[29/29] Processing: [29].pdf

[Orchestrator] Step 1: Parsing and merging paper 'c:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\backend\papers\research_papers\[29].pdf'...
[2026-08-25 01:10:07] [INFO] [paper_to_project] 🚀 Starting routed extraction pipeline for '[29].pdf'...
[2026-08-25 01:10:07] [INFO] [paper_to_project] Routing '[29].pdf' to PyMuPDF text & section parser.
[2026-08-25 01:10:07] [INFO] [paper_to_project] Extracting blocks from PDF '[29].pdf' using tw

[WARNING] 2026-08-25 01:10:51,029 [RapidOCR] main.py:132: The text detection result is empty
RapidOCR returned empty result!
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


[2026-08-25 01:10:51] [INFO] [paper_to_project] Docling conversion completed successfully for '[29].pdf'.
[2026-08-25 01:10:51] [INFO] [paper_to_project] [FINISH] Finished routed extraction for '[29].pdf'. Selected: ['pymupdf', 'docling']
[2026-08-25 01:10:51] [INFO] [paper_to_project] Merging extraction outputs for '[29].pdf' into canonical PaperDocument...
[Orchestrator] Slicing document and saving embeddings in pgvector database...
[DB WARN] PostgreSQL database unreachable. Falling back to local JSON storage: c:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\backend\papers\in_memory_vector_db.json
[DB] Local JSON database initialized successfully (Fallback).
[DB] Saved 24 chunks with local vector lists for 'paper_29'.
Initializing ChatOllama with model 'qwen2.5-coder:1.5b'...
Sending request to local Ollama for structured metadata extraction...

[Orchestrator] Step 2: Running Method Decomposition Agent...
[Decomposition Agent] Querying lo

## Cell 4: Per-Paper Summary Table

In [5]:
header = f"{'#':<4} {'PDF':<12} {'STATUS':<9} {'TIME(s)':<9} {'COMPS':<6} {'EDGES':<6} {'FEASIBILITY':<13} {'EXPLICIT':<10} {'MISSING':<8} TITLE"
print(header)
print('-' * 130)
for i, r in enumerate(all_results, 1):
    gc = r.get('gap_counts', {})
    print(
        f"{i:<4} {r['pdf_name']:<12} {'OK':<9} {r['elapsed_seconds']:<9} "
        f"{r['components_count']:<6} {r['edges_count']:<6} {r['feasibility_status']:<13} "
        f"{gc.get('EXPLICIT',0):<10} {gc.get('MISSING',0):<8} {r['title'][:45]}"
    )
for r in all_errors:
    print(
        f"{'--':<4} {r['pdf_name']:<12} {'ERROR':<9} {r['elapsed_seconds']:<9} "
        f"{'--':<6} {'--':<6} {'N/A':<13} {'--':<10} {'--':<8} {r['error'][:50]}"
    )
print()
print(f'Total: {len(all_results)} success / {len(all_errors)} errors / {len(papers_to_run)} total')

#    PDF          STATUS    TIME(s)   COMPS  EDGES  FEASIBILITY   EXPLICIT   MISSING  TITLE
----------------------------------------------------------------------------------------------------------------------------------
1    [1].pdf      OK        446.34    2      0      FEASIBLE      0          0        A Novel Change Detection Method Based on Visu
2    [2].pdf      OK        234.13    1      0      FEASIBLE      0          0        A New Learning Paradigm for Foundation Model-
3    [3].pdf      OK        243.45    1      0      FEASIBLE      0          0        ChangeCLIP: Remote sensing change detection w
4    [4].pdf      OK        439.48    1      0      FEASIBLE      0          0        Change Knowledge-Guided Vision-Language Remot
5    [5].pdf      OK        465.15    2      0      FEASIBLE      0          0        MDS-Net: An Image-Text Enhanced Multimodal Ne
6    [7].pdf      OK        1115.83   1      0      FEASIBLE      0          0        RFHP-CD: A Prompt-Driven Fine-T

## Cell 5: Corpus-Wide Aggregate Statistics

In [6]:
if all_results:
    all_feasibility  = Counter(r['feasibility_status']  for r in all_results)
    all_gap_counts   = Counter()
    all_param_status = Counter()
    total_components = sum(r['components_count'] for r in all_results)
    total_edges      = sum(r['edges_count'] for r in all_results)
    avg_elapsed      = round(sum(r['elapsed_seconds'] for r in all_results) / len(all_results), 1)
    for r in all_results:
        all_gap_counts   += Counter(r.get('gap_counts', {}))
        all_param_status += Counter(r.get('param_status_counts', {}))
    critical_missing_count = sum(1 for r in all_results if r.get('has_critical_missing'))

    print('=' * 55)
    print('CORPUS-WIDE AGGREGATE STATISTICS')
    print('=' * 55)
    print(f'  System            : {gpu_model}')
    print(f'  VRAM              : {vram_gb} GB | RAM: {system_ram_gb} GB')
    print(f'  Model             : {MODEL_NAME}')
    print()
    print(f'  Papers processed  : {len(all_results)} / {len(papers_to_run)}')
    print(f'  Papers failed     : {len(all_errors)}')
    print(f'  Avg time / paper  : {avg_elapsed}s')
    print()
    print(f'  Total components  : {total_components}')
    print(f'  Total edges       : {total_edges}')
    print(f'  Avg comps / paper : {round(total_components/len(all_results),1)}')
    print(f'  Avg edges / paper : {round(total_edges/len(all_results),1)}')
    print()
    print('  --- Feasibility Distribution ---')
    for k, v in sorted(all_feasibility.items()):
        print(f'    {k:<15}: {v} paper(s)')
    print()
    print('  --- Gap Classification Distribution ---')
    for k, v in sorted(all_gap_counts.items()):
        print(f'    {k:<15}: {v} parameters')
    print(f'  Critical missing in {critical_missing_count} paper(s)')
    print()
    print('  --- Parameter Status Distribution ---')
    for k, v in sorted(all_param_status.items()):
        print(f'    {k:<15}: {v} parameters')

CORPUS-WIDE AGGREGATE STATISTICS
  System            : NVIDIA GeForce RTX 5050 Laptop GPU
  VRAM              : 8.0 GB | RAM: 23.6 GB
  Model             : qwen2.5-coder:1.5b

  Papers processed  : 24 / 29
  Papers failed     : 5
  Avg time / paper  : 401.2s

  Total components  : 42
  Total edges       : 10
  Avg comps / paper : 1.8
  Avg edges / paper : 0.4

  --- Feasibility Distribution ---
    FEASIBLE       : 21 paper(s)
    FEASIBLE_WITH_MODIFICATION: 3 paper(s)

  --- Gap Classification Distribution ---
    AMBIGUOUS      : 66 parameters
  Critical missing in 0 paper(s)

  --- Parameter Status Distribution ---
    EXPLICIT       : 264 parameters


## Cell 6: Save Consolidated Report (Markdown + JSON)

In [7]:
import os
import datetime
import json

# Setup output paths for Phase 1 to 7
report_md_path = os.path.join(REPORTS_DIR, f'Phase_1_to_7.md')
report_json_path = os.path.join(REPORTS_DIR, f'Phase_1_to_7.json')

total_run_time = round(time.time() - run_start_time, 2) if 'run_start_time' in globals() else 0.0
avg_elapsed = round(sum(r['elapsed_seconds'] for r in all_results) / len(all_results), 1) if all_results else 0
total_components = sum(r['components_count'] for r in all_results)
total_edges = sum(r['edges_count'] for r in all_results)
all_feasibility = Counter(r['feasibility_status'] for r in all_results)

all_gap_counts = Counter()
all_resource_tiers = Counter()
critical_missing_count = 0
total_files_generated = 0

for r in all_results:
    all_gap_counts += Counter(r.get('gap_counts', {}))
    total_files_generated += r.get('generated_files_count', 0)
    res_est = r['_result_full'].get('resource_estimation') if '_result_full' in r else None
    if res_est:
        all_resource_tiers[res_est.overall_resource_tier] += 1
    if r.get('gap_counts', {}).get('MISSING', 0) > 0:
        critical_missing_count += 1

all_param_status = Counter()
for r in all_results:
    all_param_status += Counter(r.get('param_status_counts', {}))

# ==============================================================
#  MARKDOWN REPORT GENERATION
# ==============================================================
md = []
md.append('# Paper-to-Project: Phase 1–7 Corpus-Wide Code Synthesis Report')
md.append(f'**Generated:** {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
md.append(f'**Model:** `{MODEL_NAME}`' if 'MODEL_NAME' in globals() else '**Model:** `qwen2.5-coder:1.5b`')
md.append(f'**GPU:** {gpu_model} ({vram_gb} GB VRAM) | **RAM:** {system_ram_gb} GB' if 'gpu_model' in globals() else '')
md.append(f'**Total Running Time:** {total_run_time} seconds')
md.append(f'**Papers Run:** {len(papers_to_run)} | **Success:** {len(all_results)} | **Errors:** {len(all_errors)}' if 'papers_to_run' in globals() else '')
md.append('')

# Summary table
md.append('## Per-Paper Results Summary')
md.append('| # | PDF | Status | Time(s) | Comps | VRAM Est | Feasibility | Adaptations | Code Files | Title |')
md.append('|---|-----|--------|---------|-------|----------|-------------|-------------|------------|-------|')
for i, r in enumerate(all_results, 1):
    res_est = r['_result_full'].get('resource_estimation') if '_result_full' in r else None
    vram_str = f"{res_est.training.vram_recommended_gb:.1f} GB" if res_est else "N/A"
    adaptations_count = len(r['adaptations'])
    md.append(
        f"| {i} | {r['pdf_name']} | OK | {r['elapsed_seconds']} | "
        f"{r['components_count']} | {vram_str} | {r['feasibility_status']} | "
        f"{adaptations_count} | {r['generated_files_count']} | {r['title'][:50].replace('|','-')} |"
    )
md.append('')

# Per-paper details section
md.append('## Per-Paper Code Generation Blueprints')
md.append('')
for i, r in enumerate(all_results, 1):
    res  = r['_result_full']
    cg   = res.get('component_graph')
    grpt = res.get('gap_report')
    res_est = res.get('resource_estimation')
    feat = res.get('feasibility_report')
    bseq = res.get('build_sequence')
    spec = res.get('project_specification')
    tree = res.get('project_tree')
    arpt = res.get('report')

    md.append(f'### [{i}] {r["pdf_name"]} — {r["title"][:80]}')
    md.append('')
    md.append(f'- **Time:** {r["elapsed_seconds"]}s | **Feasibility:** {r["feasibility_status"]}')
    
    if spec:
        md.append(f'- **Architecture:** {spec.architecture}')
        md.append(f'- **System Requirements:** {spec.requirements}')
        
    if tree:
        md.append('\n**ASCII Project Structure Layout (Day 29):**')
        md.append('```text')
        md.append(tree.tree_structure)
        md.append('```')
        
        md.append('\n**Generated Python Modules Map (Day 30):**')
        md.append('| Generated Relative Path | Functional Module Summary |')
        md.append('|-------------------------|--------------------------|')
        for filepath, desc in tree.files.items():
            md.append(f"| {filepath} | {desc} |")
        md.append('')
    md.append('---')

with open(report_md_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(md))
print(f'[OK] Markdown report : {report_md_path}')


# ==============================================================
#  JSON REPORT SERIALIZATION
# ==============================================================
def safe_serialize(r):
    res  = r.get('_result_full', {})
    cg   = res.get('component_graph')
    grpt = res.get('gap_report')
    res_est = res.get('resource_estimation')
    feat = res.get('feasibility_report')
    bseq = res.get('build_sequence')
    extp = res.get('extracted_parameters')
    spec = res.get('project_specification')
    tree = res.get('project_tree')
    pdoc = res.get('paper_doc')
    arpt = res.get('report')
    meta = res.get('metadata')
    
    components_list = []
    if feat:
        cf_list = getattr(feat, 'components_analysis', None)
        if cf_list is None:
            cf_list = getattr(feat, 'components', [])
        if cf_list:
            for cf in cf_list:
                if isinstance(cf, dict):
                    components_list.append({
                        'name': cf.get('component_name', cf.get('name', 'N/A')),
                        'status': cf.get('status', 'N/A'),
                        'reason': cf.get('reason', 'N/A')
                    })
                else:
                    components_list.append({
                        'name': getattr(cf, 'component_name', getattr(cf, 'name', 'N/A')),
                        'status': getattr(cf, 'status', 'N/A'),
                        'reason': getattr(cf, 'reason', 'N/A')
                    })

    return {
        'paper_id'        : r['paper_id'],
        'pdf_name'        : r['pdf_name'],
        'status'          : r['status'],
        'elapsed_seconds' : r['elapsed_seconds'],
        'metadata': {
            'title'   : meta.title    if meta else 'N/A',
            'authors' : meta.authors  if meta else [],
            'abstract': meta.abstract[:500] if meta else '',
            'primary_contribution': meta.primary_contribution if meta else ''
        },
        'paper_doc_stats': {
            'sections' : len(pdoc.sections)  if pdoc else 0,
            'tables'   : len(pdoc.tables)    if pdoc else 0,
            'equations': len(pdoc.equations) if pdoc else 0
        },
        'component_graph': {
            'components': [{'name': c.name, 'type': c.type, 'params': list(c.parameters.keys())} for c in cg.components] if cg else [],
            'edges'     : cg.edges if cg else []
        },
        'extracted_parameters': {
            field: {'value': getattr(extp, field).value, 'status': getattr(extp, field).status, 'confidence': getattr(extp, field).confidence}
            for field in extp.__class__.model_fields.keys()
        } if extp else {},
        'gap_report': {
            'summary'             : grpt.summary if grpt else '',
            'has_critical_missing': grpt.has_critical_missing_parameters if grpt else None,
            'gaps': [{'parameter': g.parameter_name, 'classification': g.classification, 'value': g.value, 'details': g.details} for g in grpt.parameter_gaps] if grpt else []
        },
        'resource_estimation': {
            'param_count_millions': res_est.model.param_count_millions if res_est else 0.0,
            'weights_mb': res_est.model.model_weights_mb if res_est else 0.0,
            'vram_recommended_gb': res_est.training.vram_recommended_gb if res_est else 0.0,
            'overall_resource_tier': res_est.overall_resource_tier if res_est else 'UNKNOWN'
        } if res_est else {},
        'feasibility': {
            'overall_status'     : feat.overall_status      if feat else 'N/A',
            'training_status'    : feat.training_status     if feat else 'N/A',
            'training_substitute': feat.training_substitute if feat else '',
            'components': components_list
        },
        'project_specification': {
            'requirements': spec.requirements if spec else '',
            'architecture': spec.architecture if spec else '',
            'components': spec.components if spec else []
        } if spec else {},
        'project_tree': {
            'files_count': len(tree.files) if tree else 0,
            'structure': tree.tree_structure if tree else ''
        } if tree else {}
    }

json_report = {
    'generated_at'    : datetime.datetime.now().isoformat(),
    'system': {
        'model'         : MODEL_NAME if 'MODEL_NAME' in globals() else 'qwen2.5-coder:1.5b',
        'gpu'           : gpu_model if 'gpu_model' in globals() else 'CPU Only',
        'vram_gb'       : vram_gb if 'vram_gb' in globals() else 0.0,
        'system_ram_gb' : system_ram_gb if 'system_ram_gb' in globals() else 16.0,
        'timeline_weeks': TIMELINE_WEEKS if 'TIMELINE_WEEKS' in globals() else 2
    },
    'papers_total'    : len(papers_to_run) if 'papers_to_run' in globals() else len(all_results),
    'papers_success'  : len(all_results),
    'papers_error'    : len(all_errors),
    'total_runtime_sec': total_run_time,
    'aggregate': {
        'total_components'     : total_components if all_results else 0,
        'total_edges'          : total_edges      if all_results else 0,
        'avg_elapsed_seconds'  : avg_elapsed      if all_results else 0,
        'feasibility_dist'     : dict(all_feasibility)  if all_results else {},
        'gap_class_dist'       : dict(all_gap_counts)   if all_results else {},
        'param_status_dist'    : dict(all_param_status) if all_results else {},
        'resource_tier_dist'   : dict(all_resource_tiers) if all_results else {},
        'total_files_generated': total_files_generated,
        'critical_missing_count': critical_missing_count if all_results else 0
    },
    'papers': [safe_serialize(r) for r in all_results],
    'errors': all_errors
}

with open(report_json_path, 'w', encoding='utf-8') as f:
    json.dump(json_report, f, indent=2, ensure_ascii=False)
print(f'[OK] JSON report     : {report_json_path}')

print()
print('============================')
print('ALL PHASE 1-7 REPORTS SAVED')
print('============================')


[OK] Markdown report : c:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\backend\tests\reports\Phase_1_to_7.md
[OK] JSON report     : c:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\backend\tests\reports\Phase_1_to_7.json

ALL PHASE 1-7 REPORTS SAVED


## Cell 7: Final Scorecard

In [8]:
print('=' * 65)
print('PHASE 1-7 CORPUS SCORECARD')
print('=' * 65)
print(f"  GPU              : {gpu_model}")
print(f"  VRAM             : {vram_gb} GB | RAM: {system_ram_gb} GB")
print(f"  Model            : {MODEL_NAME}")
print()
print(f"  Papers total     : {len(papers_to_run)}")
print(f"  Success          : {len(all_results)}")
print(f"  Errors           : {len(all_errors)}")
if all_errors:
    print(f"  Failed           : {', '.join(e['pdf_name'] for e in all_errors)}")
print()
if all_results:
    print(f"  Avg time/paper   : {avg_elapsed}s")
    print(f"  Total components : {total_components}")
    print(f"  Total edges      : {total_edges}")
    print(f"  Total Py Files   : {total_files_generated} files generated")
    print()
    print(f"  --- Feasibility Status Dist ---")
    for k, v in sorted(all_feasibility.items()):
        pct = round(v / len(all_results) * 100)
        print(f"    {k:<27}: {v:>3} ({pct}%)")
    print()
    print(f"  --- Resource footprint Tiers ---")
    for k, v in sorted(all_resource_tiers.items()):
        pct = round(v / len(all_results) * 100)
        print(f"    {k:<27}: {v:>3} ({pct}%)")
print()
print(f"  Reports saved to : {REPORTS_DIR}")
print(f"    MD   : Phase_1_to_7.md")
print(f"    JSON : Phase_1_to_7.json")
print('=' * 65)
print('DONE')
print('=' * 65)


PHASE 1-7 CORPUS SCORECARD
  GPU              : NVIDIA GeForce RTX 5050 Laptop GPU
  VRAM             : 8.0 GB | RAM: 23.6 GB
  Model            : qwen2.5-coder:1.5b

  Papers total     : 29
  Success          : 24
  Errors           : 5
  Failed           : [6].pdf, [8].pdf, [16].pdf, [19].pdf, [28].pdf

  Avg time/paper   : 401.2s
  Total components : 42
  Total edges      : 10
  Total Py Files   : 125 files generated

  --- Feasibility Status Dist ---
    FEASIBLE                   :  21 (88%)
    FEASIBLE_WITH_MODIFICATION :   3 (12%)

  --- Resource footprint Tiers ---
    LOW                        :  24 (100%)

  Reports saved to : c:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\backend\tests\reports
    MD   : Phase_1_to_7.md
    JSON : Phase_1_to_7.json
DONE
